#<font color="Green">**Notebook Purpose**</font>

This notebook implements a sensitivity analysis to address Reviewer 1's Comment 3 (and Referee 3's related comment on selection bias from post-hoc cluster exclusion). The original analysis clustered all 18,652 patients into 40 clusters, then excluded 10 clusters where >70% of patients had missing longitudinal medication data. This notebook reverses that order: it applies a **patient-level** sparsity filter first (removing patients whose 12-bin trajectory has >70% empty bins), then re-clusters the remaining patients using the same Ward's linkage method with k=40.

The goal is to demonstrate that the broad therapy group categories (Monotherapy, Dual Therapy, Complex Therapy, GLP-1 Therapy, Variant Therapy) are robust to the filtering approach, by comparing original and new group assignments via a concordance/confusion matrix.

---

###<font color="Red"> Required Data </font>

1. **`patient_vectors.pkl`** — dict mapping `patient_id` → 108-dimensional trajectory vector (from `MedicationTrajectoryRepresentations.ipynb`)
2. **`patient_bins.pkl`** — dict mapping `patient_id` → list of 12 sets of medication classes (from `MedicationTrajectoryRepresentations.ipynb`)
3. **Original group pkl files** (from `ClusteringAnalysis.ipynb`):
   - `monotherapy_patients.pkl`
   - `dual_therapy_patients.pkl`
   - `complex_therapy_patients.pkl`
   - `GLP_1_therapy_patients.pkl`
   - `variant_therapy_patients.pkl`
   - `early_dropout_patients.pkl`
   - `sorted_cluster_patient_ids.pkl` — dict mapping original cluster ID → list of patient IDs (needed for cluster-level centroid matching)

## Section 1 — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pickle

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

from sklearn.cluster import AgglomerativeClustering

In [ ]:
with open('/content/patient_vectors.pkl', 'rb') as f:
    patient_vectors = pickle.load(f)

with open('/content/patient_bins.pkl', 'rb') as f:
    patient_bins = pickle.load(f)

# Original group assignments
with open('/content/monotherapy_patients.pkl', 'rb') as f:
    monotherapy_patients = pickle.load(f)

with open('/content/dual_therapy_patients.pkl', 'rb') as f:
    dual_therapy_patients = pickle.load(f)

with open('/content/complex_therapy_patients.pkl', 'rb') as f:
    complex_therapy_patients = pickle.load(f)

with open('/content/GLP_1_therapy_patients.pkl', 'rb') as f:
    GLP_1_therapy_patients = pickle.load(f)

with open('/content/variant_therapy_patients.pkl', 'rb') as f:
    variant_therapy_patients = pickle.load(f)

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pickle.load(f)

print(f'Total patients in patient_vectors: {len(patient_vectors):,}')
print(f'Total patients in patient_bins: {len(patient_bins):,}')
with open('/content/sorted_cluster_patient_ids.pkl', 'rb') as f:
    sorted_cluster_patient_ids = pickle.load(f)

print(f'Original clusters: {len(sorted_cluster_patient_ids)}')

In [ ]:
# Build original group lookup: patient_id -> group name
original_group_lookup = {}

group_dicts = {
    'Monotherapy': monotherapy_patients,
    'Dual Therapy': dual_therapy_patients,
    'Complex Therapy': complex_therapy_patients,
    'GLP-1 Therapy': GLP_1_therapy_patients,
    'Variant Therapy': variant_therapy_patients,
    'Early Dropout': early_dropout_patients,
}

for group_name, cluster_dict in group_dicts.items():
    for cluster_id, patient_list in cluster_dict.items():
        for pid in patient_list:
            original_group_lookup[pid] = group_name

# The 9,327 eligible patients from the original analysis
original_eligible_pids = set(
    pid for pid, group in original_group_lookup.items() if group != 'Early Dropout'
)

original_dropout_set = set(
    pid for pid, group in original_group_lookup.items() if group == 'Early Dropout'
)

print(f'Original eligible patients: {len(original_eligible_pids):,}')
print(f'Original early dropout patients: {sum(len(v) for v in early_dropout_patients.values()):,}')

# Build original CLUSTER -> group mapping (for centroid matching)
original_cluster_group = {}
for group_name, cluster_dict in group_dicts.items():
    for cluster_id in cluster_dict.keys():
        original_cluster_group[cluster_id] = group_name

print(f'Original cluster-to-group mappings: {len(original_cluster_group)}')
for g in ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy', 'Early Dropout']:
    ids = sorted([k for k,v in original_cluster_group.items() if v == g])
    print(f'  {g}: clusters {ids}')

## Section 2 — Patient-Level Sparsity Filter

Apply a patient-level version of the manuscript's 70% sparsity rule. A patient is excluded if more than 70% of their 12 half-year bins contain no medication data (i.e., the bin is `{'nothing'}`).

This is the direct patient-level equivalent of the original cluster-level filter: instead of clustering first and then removing clusters with >70% missing data, we remove individual patients with >70% missing data and then cluster the remainder.

In [ ]:
SPARSITY_THRESHOLD = 0.70
NUM_BINS = 12

def compute_patient_sparsity(bins):
    """Fraction of 12 bins that are empty ({'nothing'})."""
    empty = sum(1 for b in bins if b == {'nothing'} or b == set())
    return empty / NUM_BINS

# Compute sparsity for every patient
patient_sparsity = {
    pid: compute_patient_sparsity(bins)
    for pid, bins in patient_bins.items()
}

# Split into retained and excluded
retained_pids = [pid for pid, s in patient_sparsity.items() if s <= SPARSITY_THRESHOLD]
excluded_pids = [pid for pid, s in patient_sparsity.items() if s > SPARSITY_THRESHOLD]

print(f'Total patients: {len(patient_sparsity):,}')
print(f'Retained (sparsity <= {SPARSITY_THRESHOLD:.0%}): {len(retained_pids):,}')
print(f'Excluded (sparsity > {SPARSITY_THRESHOLD:.0%}): {len(excluded_pids):,}')

In [ ]:
# Median non-zero intervals: disengagement vs. primary cohort
def count_nonempty_bins(pid):
    return sum(1 for b in patient_bins[pid] if b != {'nothing'} and b != set())

# Original groups
dropout_nonempty = [count_nonempty_bins(pid) for pid in original_dropout_set if pid in patient_bins]
eligible_nonempty = [count_nonempty_bins(pid) for pid in original_eligible_pids if pid in patient_bins]

print('Non-zero medication intervals (out of 12):')
print()
print(f'  Early disengagement clusters (n={len(dropout_nonempty):,}):')
print(f'    Median: {np.median(dropout_nonempty):.0f}')
print(f'    IQR:    {np.percentile(dropout_nonempty, 25):.0f} – {np.percentile(dropout_nonempty, 75):.0f}')
print()
print(f'  Primary analytic cohort (n={len(eligible_nonempty):,}):')
print(f'    Median: {np.median(eligible_nonempty):.0f}')
print(f'    IQR:    {np.percentile(eligible_nonempty, 25):.0f} – {np.percentile(eligible_nonempty, 75):.0f}')

In [ ]:
# --- Comparison with original early dropout group ---

original_dropout_set = set(
    pid for pids in early_dropout_patients.values() for pid in pids
)
new_excluded_set = set(excluded_pids)
new_retained_set = set(retained_pids)

overlap_excluded = original_dropout_set & new_excluded_set
dropout_now_retained = original_dropout_set - new_excluded_set
eligible_now_excluded = original_eligible_pids & new_excluded_set
eligible_still_retained = original_eligible_pids & new_retained_set

print('=== Comparing Patient-Level Filter vs. Original Cluster-Level Filter ===')
print()
print(f'Original early dropout patients:              {len(original_dropout_set):,}')
print(f'New excluded patients (patient-level filter):  {len(new_excluded_set):,}')
print()
print(f'Overlap (excluded in both):                   {len(overlap_excluded):,}')
print(f'Originally dropout, now RETAINED:              {len(dropout_now_retained):,}')
print(f'Originally eligible, now EXCLUDED:             {len(eligible_now_excluded):,}')
print(f'Originally eligible, still retained:           {len(eligible_still_retained):,}')
print()
print(f'Jaccard similarity of excluded sets: '
      f'{len(overlap_excluded) / len(original_dropout_set | new_excluded_set):.3f}')

In [ ]:
# --- Sparsity distribution histogram ---

sparsity_vals = list(patient_sparsity.values())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sparsity_vals, bins=13, edgecolor='black', alpha=0.7)
ax.axvline(SPARSITY_THRESHOLD, color='red', linestyle='--',
           label=f'Threshold ({SPARSITY_THRESHOLD:.0%})')
ax.set_xlabel('Patient Sparsity (fraction of empty bins)')
ax.set_ylabel('Number of Patients')
ax.set_title('Distribution of Patient-Level Medication Sparsity')
ax.legend()
plt.tight_layout()
plt.show()

## Section 3 — Re-Clustering

Apply Ward's linkage agglomerative clustering with k=30, k=40.

In [ ]:
# Build filtered matrix
filtered_patient_ids = sorted(retained_pids)  # sort for reproducibility
X_filtered = np.array([patient_vectors[pid] for pid in filtered_patient_ids])

print(f'Clustering {len(filtered_patient_ids):,} patients with {X_filtered.shape[1]} features')

clustering = AgglomerativeClustering(
    n_clusters=40,
    linkage='ward',
    metric='euclidean'
)

clustering.fit(X_filtered)
print('Clustering complete.')

In [ ]:
# Build cluster dictionaries (sorted by size, largest = cluster 1)
cluster_results = pd.DataFrame({
    'patient_id': filtered_patient_ids,
    'cluster': clustering.labels_
})

cluster_patient_ids_raw = cluster_results.groupby('cluster')['patient_id'].apply(list).to_dict()

sorted_clusters = sorted(cluster_patient_ids_raw.items(), key=lambda x: len(x[1]), reverse=True)

new_cluster_patient_ids = {}
for i, (_, patients) in enumerate(sorted_clusters):
    new_cluster_patient_ids[i + 1] = patients

print(f'Created {len(new_cluster_patient_ids)} clusters.')
print(f'Cluster sizes: min={min(len(v) for v in new_cluster_patient_ids.values())}, '
      f'max={max(len(v) for v in new_cluster_patient_ids.values())}, '
      f'median={np.median([len(v) for v in new_cluster_patient_ids.values()]):.0f}')

## Section 4 — Cluster Summary Table

Quick overview of each new cluster: size and dominant drug classes in the first bin (2019H1). Use this as a cheat sheet when reviewing plots and assigning therapy groups.

In [ ]:
def get_top_drugs_bin1(patient_ids, patient_bins, top_n=3):
    """Return top drug classes by prevalence in bin 0 (2019H1)."""
    drug_counts = {}
    for pid in patient_ids:
        for drug in patient_bins[pid][0]:
            if drug != 'nothing':
                drug_counts[drug] = drug_counts.get(drug, 0) + 1
    total = len(patient_ids)
    sorted_drugs = sorted(drug_counts.items(), key=lambda x: x[1], reverse=True)
    return ', '.join(f'{d} ({c/total*100:.0f}%)' for d, c in sorted_drugs[:top_n])


summary_rows = []
for cid in sorted(new_cluster_patient_ids.keys()):
    pids = new_cluster_patient_ids[cid]
    top_drugs = get_top_drugs_bin1(pids, patient_bins)
    summary_rows.append({
        'Cluster': cid,
        'N Patients': len(pids),
        'Top Drugs (2019H1)': top_drugs if top_drugs else 'none'
    })

cluster_summary_df = pd.DataFrame(summary_rows)
cluster_summary_df

## Section 5 — Visualization Functions & Cluster Viewer

Two-panel visualizations for each cluster: (1) patient-level medication heatmap, (2) medication distribution line chart. These are the same functions from `ClusterLevelPlotsCreation.ipynb`, reproduced here for self-containment.

In [ ]:
PRESCRIPTION_COLORS = {
    'SGLT2': '#00FF00',
    'SUL': '#FFB6C1',
    'Insulin': '#FF0000',
    'MET': '#0000FF',
    'DPP-4': '#8B4513',
    'GLP-1': '#FFDB58',
    'GIP/GLP-1': '#40E0D0',
    'TZD': '#FF8C00',
    'Other': '#000000',
    'nothing': '#FFFFFF'
}

def mix_colors(colors):
    rgb_colors = np.array([to_rgb(PRESCRIPTION_COLORS[color])
                           for color in colors if color in PRESCRIPTION_COLORS])
    if len(rgb_colors) == 0:
        return np.array(to_rgb(PRESCRIPTION_COLORS['nothing']))
    return np.mean(rgb_colors, axis=0).astype(np.float32)


def plot_cluster_heatmap(ax, cluster_patient_ids, patient_bins_cluster):
    num_bins = 12
    heatmap_data = np.ones((len(cluster_patient_ids), num_bins, 3), dtype=np.float32)
    for i, patient_id in enumerate(cluster_patient_ids):
        for j, prescriptions in enumerate(patient_bins_cluster[patient_id]):
            heatmap_data[i, j] = mix_colors(prescriptions)
    ax.imshow(heatmap_data, aspect='auto', interpolation='none')
    ax.set_xticks(range(num_bins))
    labels = [f'{year}' if h == 1 else '' for year in range(2019, 2025) for h in (1, 2)]
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks([])
    ax.set_title('1. Individual Medication Sequences', fontsize=14)


def plot_medication_distribution(ax, cluster_patient_ids, patient_bins_cluster):
    num_bins = 12
    drug_classes = [d for d in PRESCRIPTION_COLORS.keys() if d != 'nothing']
    drug_counts = {drug: [0]*num_bins for drug in drug_classes}
    total_patients = len(cluster_patient_ids)
    for patient_id in cluster_patient_ids:
        for bin_index, prescriptions in enumerate(patient_bins_cluster[patient_id]):
            for drug in prescriptions:
                if drug in drug_counts:
                    drug_counts[drug][bin_index] += 1
    for drug in drug_counts:
        drug_counts[drug] = [count / total_patients * 100 for count in drug_counts[drug]]
    for drug, percentages in drug_counts.items():
        ax.plot(range(num_bins), percentages, label=drug, color=PRESCRIPTION_COLORS[drug])
    ax.set_ylim(0, 100)
    ax.set_xticks(range(num_bins))
    labels = [f'{year}' if h == 1 else '' for year in range(2019, 2025) for h in (1, 2)]
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title('2. % Taking Each Medication Class', fontsize=14)
    ax.grid(True)
    ax.set_ylabel('')
    ax.legend(fontsize=7, loc='upper right')

In [ ]:
def view_cluster(cluster_id):
    """Display the 2-panel visualization for a given cluster."""
    if cluster_id not in new_cluster_patient_ids:
        print(f'Cluster {cluster_id} not found. Valid IDs: {sorted(new_cluster_patient_ids.keys())}')
        return

    pids = new_cluster_patient_ids[cluster_id]
    patient_bins_cluster = {pid: patient_bins[pid] for pid in pids}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    plot_cluster_heatmap(axes[0], pids, patient_bins_cluster)
    plot_medication_distribution(axes[1], pids, patient_bins_cluster)

    fig.suptitle(f'Sensitivity Analysis — Cluster {cluster_id} (n={len(pids):,})',
                 fontsize=16, fontweight='bold')
    fig.subplots_adjust(left=0.05, right=0.97, top=0.85, bottom=0.12, wspace=0.18)
    plt.show()


# Example: view the largest cluster
view_cluster(1)

In [ ]:
# View all clusters sequentially (uncomment to run)
# for cid in sorted(new_cluster_patient_ids.keys()):
#     view_cluster(cid)

## Section 6 — Automated Therapy Group Assignment via Cluster-Level Centroid Matching

Rather than manually assigning each of the 40 new clusters to therapy groups, we automate this using the original expert-curated cluster assignments as reference. The approach:

1. Compute a **medication profile centroid** for each of the 30 original non-dropout clusters (a 108-dimensional vector where each dimension is the percentage of patients in that cluster taking drug class *j* in bin *t*).
2. Compute the same profile for each of the 40 new clusters.
3. For each new cluster, find its **nearest original cluster** by cosine similarity.
4. Assign the new cluster to the therapy group of its nearest original cluster.
5. Flag any assignment where the top match and second-best match (from a *different* group) are within a similarity margin of 0.05, for optional manual review.

This inherits the clinical judgment that went into the original groupings without requiring a second round of expert review.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

DRUG_CLASSES = ['MET', 'SUL', 'SGLT2', 'GLP-1', 'GIP/GLP-1', 'Insulin', 'DPP-4', 'TZD', 'Other']
NUM_BINS = 12
CONFIDENCE_MARGIN = 0.05  # flag if top two groups are within this margin

def compute_cluster_profile(patient_ids, patient_bins):
    """
    Compute the medication profile centroid for a cluster.
    Returns a 108-dim vector: for each of 12 bins, the percentage of
    patients taking each of 9 drug classes.
    """
    n = len(patient_ids)
    if n == 0:
        return np.zeros(NUM_BINS * len(DRUG_CLASSES))

    profile = np.zeros((NUM_BINS, len(DRUG_CLASSES)))
    for pid in patient_ids:
        bins = patient_bins[pid]
        for bin_idx, bin_set in enumerate(bins):
            for drug_idx, drug in enumerate(DRUG_CLASSES):
                if drug in bin_set:
                    profile[bin_idx, drug_idx] += 1

    profile = profile / n  # convert counts to proportions
    return profile.flatten()

In [ ]:
# Step 1: Compute centroids for each ORIGINAL non-dropout cluster
original_non_dropout_ids = [
    cid for cid, group in original_cluster_group.items() if group != 'Early Dropout'
]

original_centroids = {}
for cid in original_non_dropout_ids:
    pids = sorted_cluster_patient_ids[cid]
    original_centroids[cid] = compute_cluster_profile(pids, patient_bins)

print(f'Computed centroids for {len(original_centroids)} original clusters')
print(f'Groups represented: {sorted(set(original_cluster_group[c] for c in original_centroids))}')

In [ ]:
# Step 2: Compute centroids for each NEW cluster
new_centroids = {}
for cid, pids in new_cluster_patient_ids.items():
    new_centroids[cid] = compute_cluster_profile(pids, patient_bins)

print(f'Computed centroids for {len(new_centroids)} new clusters')

In [ ]:
# Step 3: For each new cluster, find nearest original cluster + per-group best matches

# Build matrices for vectorized similarity
orig_cids = sorted(original_centroids.keys())
orig_matrix = np.array([original_centroids[cid] for cid in orig_cids])

new_cids = sorted(new_centroids.keys())
new_matrix = np.array([new_centroids[cid] for cid in new_cids])

# Cosine similarity: (n_new x n_orig)
sim_matrix = cosine_similarity(new_matrix, orig_matrix)

# For each new cluster, collect assignment info
assignment_records = []

for i, new_cid in enumerate(new_cids):
    sims = sim_matrix[i]

    # Best overall match
    best_idx = np.argmax(sims)
    best_orig_cid = orig_cids[best_idx]
    best_sim = sims[best_idx]
    assigned_group = original_cluster_group[best_orig_cid]

    # Best match PER GROUP (to see runner-up from a different group)
    group_best = {}
    for j, orig_cid in enumerate(orig_cids):
        g = original_cluster_group[orig_cid]
        if g == 'Early Dropout':
            continue
        if g not in group_best or sims[j] > group_best[g]['sim']:
            group_best[g] = {'orig_cid': orig_cid, 'sim': sims[j]}

    # Runner-up: best match from a DIFFERENT group
    runner_up_group = None
    runner_up_sim = -1
    for g, info in group_best.items():
        if g != assigned_group and info['sim'] > runner_up_sim:
            runner_up_group = g
            runner_up_sim = info['sim']

    margin = best_sim - runner_up_sim
    confident = margin > CONFIDENCE_MARGIN

    assignment_records.append({
        'New Cluster': new_cid,
        'N Patients': len(new_cluster_patient_ids[new_cid]),
        'Assigned Group': assigned_group,
        'Best Match (Orig Cluster)': best_orig_cid,
        'Similarity': round(best_sim, 4),
        'Runner-Up Group': runner_up_group,
        'Runner-Up Sim': round(runner_up_sim, 4),
        'Margin': round(margin, 4),
        'Confident': confident
    })

In [ ]:
# Step 4: Display the full assignment table
assignment_df = pd.DataFrame(assignment_records)

# Summary stats
n_confident = assignment_df['Confident'].sum()
n_flagged = len(assignment_df) - n_confident

print(f'Auto-assigned (confident, margin > {CONFIDENCE_MARGIN}): {n_confident} / {len(assignment_df)}')
print(f'Flagged for review: {n_flagged}')
print()

# Show group sizes
print('Assigned group sizes:')
group_sizes = assignment_df.groupby('Assigned Group')['N Patients'].sum()
for g in ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy']:
    if g in group_sizes.index:
        n_clusters = (assignment_df['Assigned Group'] == g).sum()
        print(f'  {g}: {group_sizes[g]:,} patients across {n_clusters} clusters')
print()

assignment_df

In [ ]:
# Step 5: Show flagged clusters (low confidence) for optional manual review
flagged = assignment_df[~assignment_df['Confident']].copy()

if len(flagged) == 0:
    print('No clusters flagged — all assignments are confident.')
else:
    print(f'{len(flagged)} cluster(s) flagged for review (margin <= {CONFIDENCE_MARGIN}):')
    print()
    print(flagged[['New Cluster', 'N Patients', 'Assigned Group', 'Similarity',
                   'Runner-Up Group', 'Runner-Up Sim', 'Margin']].to_string(index=False))
    print()
    print('Use view_cluster(cluster_id) above to inspect these clusters visually.')
    print('To override an assignment, modify the override dict below and re-run.')

In [ ]:
view_cluster(21)

In [ ]:
# Step 6: Optional manual overrides for flagged clusters
# After reviewing flagged clusters with view_cluster(), add overrides here.
# Example: overrides = {14: 'Complex Therapy', 27: 'Variant Therapy'}

overrides = {31: 'GLP-1 Therapy', 32: 'Complex Therapy' }  # <-- fill in if needed after reviewing flagged clusters

# Apply overrides
for cid, group in overrides.items():
    mask = assignment_df['New Cluster'] == cid
    old_group = assignment_df.loc[mask, 'Assigned Group'].values[0]
    assignment_df.loc[mask, 'Assigned Group'] = group
    print(f'Override: Cluster {cid} changed from {old_group} -> {group}')

if not overrides:
    print('No overrides applied. Using all automated assignments.')

In [ ]:
# Build the final group lookup and group assignment dicts (same format as original notebook)
GROUP_ORDER = ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy']

new_group_assignments = {g: [] for g in GROUP_ORDER}
for _, row in assignment_df.iterrows():
    new_group_assignments[row['Assigned Group']].append(row['New Cluster'])

new_group_lookup = {}
for group_name, cluster_ids in new_group_assignments.items():
    for cid in cluster_ids:
        for pid in new_cluster_patient_ids[cid]:
            new_group_lookup[pid] = group_name

# Print final group sizes
print('Final therapy group sizes:')
for group in GROUP_ORDER:
    n = sum(1 for v in new_group_lookup.values() if v == group)
    cids = sorted(new_group_assignments[group])
    print(f'  {group}: {n:,} patients | clusters: {cids}')

## Section 7 — Concordance / Confusion Matrix

Cross-tabulate original vs. new therapy group assignments for the patients who appear in both analyses (the original 9,327 eligible patients). Patients who were in the original Early Dropout group but are now retained are shown separately.

In [ ]:
# Identify the overlap: original eligible patients who are also in the new clustering
overlap_pids = original_eligible_pids & set(new_group_lookup.keys())

print(f'Original eligible patients:        {len(original_eligible_pids):,}')
print(f'Patients in new clustering:         {len(new_group_lookup):,}')
print(f'Overlap (in both):                  {len(overlap_pids):,}')
print(f'Original eligible, now excluded:    {len(original_eligible_pids - set(new_group_lookup.keys())):,}')
print(f'Originally dropout, now included:   {len(set(new_group_lookup.keys()) - original_eligible_pids):,}')

In [ ]:
# Build the confusion matrix for overlap patients
GROUP_ORDER = ['Monotherapy', 'Dual Therapy', 'Complex Therapy', 'GLP-1 Therapy', 'Variant Therapy']

records = []
for pid in overlap_pids:
    records.append({
        'patient_id': pid,
        'Original Group': original_group_lookup[pid],
        'New Group': new_group_lookup[pid]
    })

comparison_df = pd.DataFrame(records)

confusion = pd.crosstab(
    comparison_df['Original Group'],
    comparison_df['New Group'],
    margins=True,
    margins_name='Total'
)

# Reorder rows and columns
row_order = [g for g in GROUP_ORDER if g in confusion.index] + ['Total']
col_order = [g for g in GROUP_ORDER if g in confusion.columns] + ['Total']
confusion = confusion.reindex(index=row_order, columns=col_order, fill_value=0)

print('Concordance Matrix (Original rows × New columns):')
print()
confusion

In [ ]:
# Concordance rate per group (diagonal / row total)
print('Concordance rates (% of original group retained in same new group):')
print()
for group in GROUP_ORDER:
    if group in confusion.index and group in confusion.columns:
        diagonal = confusion.loc[group, group]
        row_total = confusion.loc[group, 'Total']
        rate = diagonal / row_total * 100 if row_total > 0 else 0
        print(f'  {group:20s}: {diagonal:,} / {row_total:,} = {rate:.1f}%')

In [ ]:
import matplotlib as mpl
from pathlib import Path

# --- Journal-quality global settings (set once per session) ---
mpl.rcParams['pdf.fonttype']    = 42       # TrueType, not Type 3
mpl.rcParams['ps.fonttype']     = 42
mpl.rcParams['font.family']     = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
mpl.rcParams['savefig.bbox']    = 'tight'
mpl.rcParams['savefig.pad_inches'] = 0.05

# --- Figure ---
CONCORDANCE_ORDER = ['Monotherapy', 'GLP-1 Therapy', 'Dual Therapy', 'Complex Therapy', 'Variant Therapy']

confusion_core = confusion.loc[CONCORDANCE_ORDER, CONCORDANCE_ORDER].copy()
confusion_pct = confusion_core.div(confusion_core.sum(axis=1), axis=0) * 100

# ~140 mm wide — fits between single and double column
fig, ax = plt.subplots(figsize=(5.5, 4.5))

sns.heatmap(
    confusion_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
    vmin=0, vmax=100, annot_kws={'size': 9},
    cbar_kws={'label': '% of original group'}
)

ax.set_title('Patient Retention Across Therapy Groups\nPatient-Level Sparsity Filter',
             fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Sensitivity Analysis Group Assignment', fontsize=10, labelpad=8)
ax.set_ylabel('Original Group Assignment',             fontsize=10, labelpad=8)
ax.tick_params(axis='both', labelsize=9)

plt.tight_layout()

# --- Export in multiple formats ---
out_dir = Path('/content/figures'); out_dir.mkdir(exist_ok=True)
stem = 'fig_sparsity_concordance'

fig.savefig(out_dir / f'{stem}.pdf')                # vector — primary submission file
fig.savefig(out_dir / f'{stem}.eps')                # vector — alt format some systems want
fig.savefig(out_dir / f'{stem}.tif', dpi=600)       # raster — high-res TIFF
fig.savefig(out_dir / f'{stem}.png', dpi=600)       # raster — for slides / previews

plt.show()
print(f'Saved 4 versions to {out_dir}/')

In [ ]:
# Which new clusters contain original Complex Therapy patients who were assigned to Dual Therapy?
complex_to_dual = comparison_df[
    (comparison_df['Original Group'] == 'Complex Therapy') &
    (comparison_df['New Group'] == 'Dual Therapy')
]

# Break down by new cluster
cluster_breakdown = complex_to_dual['patient_id'].apply(
    lambda pid: [cid for cid, pids in new_cluster_patient_ids.items() if pid in pids][0]
)

cluster_counts = cluster_breakdown.value_counts().reset_index()
cluster_counts.columns = ['New Cluster', 'Complex→Dual Patients']

# Add total cluster size and the percentage that came from original Complex
cluster_counts['Cluster Size'] = cluster_counts['New Cluster'].apply(
    lambda cid: len(new_cluster_patient_ids[cid])
)
cluster_counts['% of Cluster from Complex'] = (
    cluster_counts['Complex→Dual Patients'] / cluster_counts['Cluster Size'] * 100
).round(1)

print(f"Total Complex→Dual patients: {len(complex_to_dual):,}")
print(f"Spread across {len(cluster_counts)} new clusters:\n")
print(cluster_counts.to_string(index=False))
print("\n# Use view_cluster(cluster_id) to inspect any of these")

## Section 8 — Export

Export new clustering results and group assignments for downstream use.

In [ ]:
# Export new cluster-patient dictionary
with open('/content/sensitivity_cluster_patient_ids.pkl', 'wb') as f:
    pickle.dump(new_cluster_patient_ids, f)

# Export new group dictionaries
for group_name, cluster_ids in new_group_assignments.items():
    group_dict = {}
    for cid in cluster_ids:
        group_dict[cid] = new_cluster_patient_ids[cid]
    safe_name = group_name.lower().replace(' ', '_').replace('-', '')
    with open(f'/content/sensitivity_{safe_name}_patients.pkl', 'wb') as f:
        pickle.dump(group_dict, f)
    print(f'Exported sensitivity_{safe_name}_patients.pkl')

# Export confusion matrix
confusion.to_excel('/content/sensitivity_concordance_matrix.xlsx')
print('Exported sensitivity_concordance_matrix.xlsx')

## Section 9 — Key Numbers for the Response Letter

Summary of the statistics to cite when responding to Reviewer 1 Comment 3 and Referee 3:

- Total patients before filtering: [X]
- Patients excluded by patient-level 70% sparsity filter: [X]
- Patients retained for re-clustering: [X]
- Of the original 9,327 eligible patients, [X] were also retained under the new filter
- Automated group assignment: [X] of 40 clusters assigned confidently (cosine similarity margin > 0.05), [X] flagged for review
- Concordance rates by group:
  - Monotherapy: [X]%
  - Dual Therapy: [X]%
  - Complex Therapy: [X]%
  - GLP-1 Therapy: [X]%
  - Variant Therapy: [X]%

**Suggested language for the response letter / supplement:**

"As a sensitivity analysis, we applied a patient-level sparsity filter (excluding patients with >70% empty medication bins) before re-clustering, rather than the post-hoc cluster-level exclusion used in the primary analysis. Re-clustering was performed using the same Ward's linkage method with k=40. New clusters were assigned to therapy groups using a nearest-centroid classifier based on cosine similarity to the 30 original expert-curated cluster profiles; [X] of 40 assignments were made with high confidence (similarity margin >0.05), and [X] were reviewed manually. Among the [X] patients present in both analyses, concordance rates ranged from [X]% (Variant Therapy) to [X]% (Monotherapy), confirming that the primary therapy group classifications are robust to the exclusion strategy."